In [29]:
%matplotlib module://ipympl.backend_nbagg
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.ndimage import maximum_filter

import sys
sys.path.insert(0, "/workspace/segmentation")
from tools import img_show3D, window, img_show2D, img_show3D_interactive, img_compare3D


In [30]:
array_3d = np.zeros((10,10,10))
shape = array_3d.shape

for z in range (shape[2]):

    for y in range (shape[1]):

        for x in range (shape[0]):

            array_3d[z, y, x] = random.randint(1, 100)

In [31]:
array_3d

array([[[ 52.,  59.,   8.,  75.,  61.,  79.,  78.,  82.,  37.,  72.],
        [ 58.,  27.,   5.,   4.,  37.,  34.,  66.,  62.,   8.,  16.],
        [  4.,  49.,  43.,  31.,  49.,  66.,  76.,  48.,  40.,  79.],
        [ 63.,  14.,  31.,  30.,   1.,  83.,  42.,  33.,  78.,  21.],
        [ 94.,  17.,  64.,  33.,  74.,   2.,  96.,  48.,  60.,  36.],
        [ 25.,  55.,  47.,  56.,  92.,  91.,  94.,  37.,  25.,  31.],
        [100.,  15.,  75.,  25.,  98.,  58.,  43.,  79.,  53.,  11.],
        [ 76.,  48.,  36.,  66.,  90.,  83.,  76.,  68.,  65.,  49.],
        [  5.,  95.,  10.,  26.,   8.,  94.,  69.,  97.,  91.,  97.],
        [ 80.,  27.,  73.,  77.,  79.,   1.,  74.,  40.,   9.,  20.]],

       [[ 18.,  12.,  26.,  79.,  84.,  18.,  67.,  15.,  56.,   4.],
        [ 66.,  68.,  44.,  10.,  53.,  98.,  29.,  82.,  21.,  63.],
        [ 99.,  27.,  20.,  10.,  77.,  24.,  24.,  79.,  86.,  62.],
        [ 25.,   5.,  49.,  77.,  47.,  78.,  44., 100.,  63.,  72.],
        [ 84.,  60

In [32]:
img_dir = Path ("/workspace/segmentation/data/practice/image_quadra/QUADRA_HC_021/retest_CT-AC.nii.gz")
img = sitk.ReadImage(img_dir)

mask_dir = Path ("/workspace/segmentation/data/practice/mask_quadra/QUADRA_HC_021/retest/liver.nii.gz")
mask = sitk.ReadImage(mask_dir)

In [33]:
img_syn = sitk.GetImageFromArray(array_3d)
img_syn.SetSpacing(img.GetSpacing())

print(img_syn.GetSpacing())

(1.5234375, 1.5234375, 2.0)


In [34]:
def cast(img):
    width, height, depth = img.GetSize()

    for y in range(height):
        for x in range(width):
            for z in range(depth):
                print(img[x, y, z], end="  ")
            print()  # Start a new line after each row.

In [35]:
cast(img_syn)  

52.0  18.0  28.0  28.0  20.0  43.0  56.0  89.0  89.0  74.0  
59.0  12.0  11.0  93.0  17.0  71.0  80.0  70.0  42.0  25.0  
8.0  26.0  88.0  79.0  43.0  67.0  7.0  53.0  86.0  74.0  
75.0  79.0  96.0  89.0  65.0  80.0  52.0  58.0  19.0  40.0  
61.0  84.0  77.0  24.0  46.0  50.0  62.0  59.0  1.0  39.0  
79.0  18.0  92.0  61.0  59.0  57.0  57.0  65.0  44.0  79.0  
78.0  67.0  87.0  53.0  35.0  15.0  95.0  75.0  23.0  47.0  
82.0  15.0  81.0  59.0  25.0  76.0  81.0  28.0  14.0  59.0  
37.0  56.0  24.0  24.0  63.0  16.0  32.0  79.0  50.0  33.0  
72.0  4.0  1.0  56.0  17.0  84.0  24.0  52.0  29.0  49.0  
58.0  66.0  93.0  51.0  61.0  32.0  2.0  47.0  41.0  18.0  
27.0  68.0  13.0  80.0  84.0  37.0  99.0  63.0  5.0  24.0  
5.0  44.0  27.0  65.0  84.0  62.0  69.0  38.0  54.0  47.0  
4.0  10.0  47.0  64.0  94.0  84.0  80.0  19.0  24.0  89.0  
37.0  53.0  11.0  75.0  12.0  74.0  54.0  35.0  75.0  47.0  
34.0  98.0  79.0  5.0  82.0  100.0  67.0  33.0  23.0  100.0  
66.0  29.0  58.0  27.0  36.0  49

In [36]:
view = img_show3D_interactive(img_syn)

In [37]:
spacing = np.asarray(img_syn.GetSpacing())
radius_mm = 1
radius_voxels = np.maximum(np.ceil(radius_mm/spacing), 1)
kernel_size = tuple(int(2 * radius + 1) for radius in radius_voxels)


In [38]:
kernel_size

(3, 3, 3)

In [39]:
img_syn_array = sitk.GetArrayFromImage(img_syn)

In [40]:
maxima = maximum_filter(img_syn_array, size= kernel_size, mode= "nearest")

In [41]:
maxima

array([[[ 68.,  68.,  79.,  84.,  98.,  98.,  98.,  82.,  82.,  72.],
        [ 99.,  99.,  79.,  84.,  98.,  98.,  98.,  86.,  86.,  86.],
        [ 99.,  99.,  77.,  77.,  98.,  98., 100., 100., 100.,  86.],
        [ 99.,  99.,  77.,  77.,  83.,  96., 100., 100., 100.,  92.],
        [ 94.,  94.,  77.,  92.,  92.,  96., 100., 100., 100.,  92.],
        [100., 100.,  93., 100., 100., 100.,  96.,  96.,  95.,  94.],
        [100., 100.,  93., 100., 100., 100.,  95.,  95.,  95.,  94.],
        [100., 100.,  95., 100., 100., 100.,  97.,  97.,  97.,  97.],
        [ 95.,  96.,  96.,  96.,  94.,  94.,  97.,  97.,  97.,  97.],
        [ 95.,  96.,  96.,  96.,  94.,  94.,  97.,  97.,  97.,  97.]],

       [[ 93.,  93.,  96.,  96.,  98.,  98.,  98.,  87.,  82.,  74.],
        [ 99.,  99.,  96.,  96.,  98.,  98.,  98.,  98.,  97.,  86.],
        [ 99.,  99.,  87.,  82.,  98.,  98., 100., 100., 100.,  86.],
        [ 99.,  99.,  87.,  99.,  99.,  99., 100., 100., 100.,  92.],
        [ 99.,  99

In [42]:
quartile = 0.9
coordinates = np.argwhere( (img_syn_array == maxima) & (img_syn_array >= quartile) )

In [43]:
selected_point_mask = np.zeros((10,10,10))

for coordinate in coordinates:
    selected_point_mask[tuple(coordinate)] = 1


selected_point_mask

array([[[0., 0., 0., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]

In [44]:
selected = sitk.GetImageFromArray(selected_point_mask)

In [45]:
compare = img_compare3D (
    selected,
    img_syn,
    name1= "selected",
    name2= "original image"
)

In [46]:
mask

<SimpleITK.SimpleITK.Image; proxy of <Swig Object of type 'itk::simple::Image *' at 0x7349be4baff0> >

In [47]:
mask_vec = sitk.GetArrayFromImage(mask)

In [48]:
mask_vec_boundary = np.asarray(mask_vec, dtype= bool)